# Session 2.1 : Data Transformation

_Analytics Through Coding Autumn 2026_

---

Visualisation and statistical analysis are only useful if the data are in the form required to answer the question.

In practice, we often need to:

* keep only relevant observations;
* reorder observations;
* select or rename variables;
* create new variables;
* group observations;
* calculate summaries.

In this session we will use the `flights` dataset, which contains flights that departed from New York City in 2013.

Rather than learning pandas functions in isolation, we will start with an **analytical question** and then decide what transformation is needed.

---

## Starting out

As always, import the necessary libraries and dataset at the start of the notebook.

In [1]:
import pandas as pd
import numpy as np

flights = pd.read_csv("../Data/nycflights13_flights.csv", index_col=0)
flights.reset_index(drop=True, inplace=True)

flights.head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
0,2013,3,25,1929.0,1905,24.0,2236.0,2217,19.0,UA,1471,N37298,EWR,RSW,169.0,1068,19,5,2013-03-25 19:00:00
1,2013,4,26,956.0,1000,-4.0,1257.0,1334,-37.0,DL,1765,N717TW,JFK,SFO,337.0,2586,10,0,2013-04-26 10:00:00
2,2013,5,21,1320.0,1309,11.0,1430.0,1414,16.0,EV,4129,N11536,EWR,DCA,39.0,199,13,9,2013-05-21 13:00:00
3,2013,7,18,1222.0,1230,-8.0,1357.0,1419,-22.0,EV,5796,N13958,EWR,CLT,77.0,529,12,30,2013-07-18 12:00:00
4,2013,8,29,540.0,545,-5.0,921.0,921,0.0,B6,939,N535JB,JFK,BQN,198.0,1576,5,45,2013-08-29 05:00:00


Before transforming a dataset, remind yourself what one row represents and check the variables available.

In [2]:
print("Shape:", flights.shape)
print(flights.columns.tolist())

Shape: (202066, 19)
['year', 'month', 'day', 'dep_time', 'sched_dep_time', 'dep_delay', 'arr_time', 'sched_arr_time', 'arr_delay', 'carrier', 'flight', 'tailnum', 'origin', 'dest', 'air_time', 'distance', 'hour', 'minute', 'time_hour']


## A small set of transformation tools

We will focus on a small number of pandas methods that can be combined to answer many analytical questions.

| pandas method | What it helps us do |
|---|---|
| `query()` | Keep observations that satisfy a condition |
| `sort_values()` | Reorder observations |
| `loc[]` | Select rows and/or columns |
| `rename()` | Rename variables |
| `assign()` | Create new variables |
| `groupby()` | Divide observations into groups |
| `agg()` | Calculate summaries |

The important skill is not memorising this table. It is recognising **which operation is required by the analytical question**.

## Question 1: Which flights departed on 16 August?

We only want observations where:

* `month == 8`
* `day == 16`

We can use `.query()` to filter observations.

In [4]:
flights_aug16 = flights.query("month == 8 and day == 16")
flights_aug16.head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
286,2013,8,16,1510.0,1455,15.0,1652.0,1701,-9.0,9E,4120,N8775A,JFK,CLE,73.0,425,14,55,2013-08-16 14:00:00
410,2013,8,16,1256.0,1255,1.0,1543.0,1545,-2.0,UA,1641,N19136,EWR,MCO,143.0,937,12,55,2013-08-16 12:00:00
707,2013,8,16,1959.0,2000,-1.0,2242.0,2310,-28.0,DL,2391,N916DL,JFK,TPA,146.0,1005,20,0,2013-08-16 20:00:00
1564,2013,8,16,619.0,620,-1.0,906.0,843,23.0,DL,1743,N6704Z,JFK,ATL,121.0,760,6,20,2013-08-16 06:00:00
1573,2013,8,16,2156.0,2159,-3.0,2258.0,2324,-26.0,UA,1116,N71411,EWR,BOS,42.0,200,21,59,2013-08-16 21:00:00


Notice that pandas returns a **new DataFrame**. The original `flights` DataFrame has not been changed.

In [6]:
print("Filtered df Shape:", flights_aug16.shape)

print("DF Shape:", flights.shape)


Filtered df Shape: (608, 19)
DF Shape: (202066, 19)


Multiple arguments to `.query()` are combined with `“and”`: every expression must be true in order for a row to be included in the output. For some operations you may need other Boolean operations - `&` is “and”, `|` is “or”, and `!` is “not”

![GitHub Codespaces](Boolean_operators.png)

### Exercise 1

Find all flights that:

* departed from `JFK`;
* travelled to `LAX`; and
* had a departure delay greater than 60 minutes.

Keep the result in a DataFrame called `jfk_lax_delayed`.

How many flights meet all three conditions?

In [14]:
jfk_lax_flights = flights.query("origin == 'JFK' and dest == 'LAX' and dep_delay > 60")
jfk_lax_flights.head()
jfk_lax_flights.shape

(359, 19)

## Question 2: Which flights experienced the largest departure delays?

Filtering determines **which observations we keep**.

Sorting determines **the order in which we inspect them**.

In [11]:
flights.sort_values("dep_delay", ascending=False).head(10)

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
114519,2013,1,9,641.0,900,1301.0,1242.0,1530,1272.0,HA,51,N384HA,JFK,HNL,640.0,4983,9,0,2013-01-09 09:00:00
74463,2013,1,10,1121.0,1635,1126.0,1239.0,1810,1109.0,MQ,3695,N517MQ,EWR,ORD,111.0,719,16,35,2013-01-10 16:00:00
176637,2013,9,20,1139.0,1845,1014.0,1457.0,2210,1007.0,AA,177,N338AA,JFK,SFO,354.0,2586,18,45,2013-09-20 18:00:00
201196,2013,3,17,2321.0,810,911.0,135.0,1020,915.0,DL,2119,N927DA,LGA,MSP,167.0,1020,8,10,2013-03-17 08:00:00
53590,2013,7,22,2257.0,759,898.0,121.0,1026,895.0,DL,2047,N6716C,LGA,ATL,109.0,762,7,59,2013-07-22 07:00:00
150167,2013,5,3,1133.0,2055,878.0,1250.0,2215,875.0,MQ,3744,N523MQ,EWR,ORD,112.0,719,20,55,2013-05-03 20:00:00
54279,2013,5,19,713.0,1700,853.0,1007.0,1955,852.0,AA,257,N3HEAA,JFK,LAS,323.0,2248,17,0,2013-05-19 17:00:00
103118,2013,1,1,848.0,1835,853.0,1001.0,1950,851.0,MQ,3944,N942MQ,JFK,BWI,41.0,184,18,35,2013-01-01 18:00:00
37737,2013,2,10,2243.0,830,853.0,100.0,1106,834.0,F9,835,N203FR,LGA,DEN,233.0,1620,8,30,2013-02-10 08:00:00
151237,2013,12,19,734.0,1725,849.0,1046.0,2039,847.0,DL,1223,N375NC,EWR,SLC,290.0,1969,17,25,2013-12-19 17:00:00


We can also sort using more than one variable.

For example, the following sorts chronologically by month and day.

In [18]:
flights.sort_values(["month","day","dep_time"], ascending=True)[["month","day","dep_time","origin", "dest"]].head(10)

,month,day,dep_time,origin,dest
49463,1,1,517.0,EWR,IAH
168799,1,1,533.0,LGA,IAH
8721,1,1,542.0,JFK,MIA
114668,1,1,554.0,LGA,ATL
58344,1,1,555.0,EWR,FLL
6821,1,1,557.0,LGA,IAD
184364,1,1,557.0,JFK,MCO
85789,1,1,558.0,JFK,LAX
90391,1,1,558.0,JFK,TPA
125702,1,1,558.0,JFK,PBI


### Exercise 2

Find the **10 flights with the longest arrival delays**.

Display only:

* `origin`
* `dest`
* `carrier`
* `arr_delay`

Sort the result from the largest delay to the smallest.

In [16]:
flights.loc[flights["arr_delay"].sort_values(ascending=False).head(10).index, ["origin", "dest", "carrier", "arr_delay"]]

,origin,dest,carrier,arr_delay
114519,JFK,HNL,HA,1272.0
74463,EWR,ORD,MQ,1109.0
176637,JFK,SFO,AA,1007.0
201196,LGA,MSP,DL,915.0
53590,LGA,ATL,DL,895.0
150167,EWR,ORD,MQ,875.0
194528,JFK,TPA,DL,856.0
54279,JFK,LAS,AA,852.0
103118,JFK,BWI,MQ,851.0
151237,EWR,SLC,DL,847.0


## Question 3: Which variables do we actually need?

Real datasets often contain many more variables than are required for a particular question.

For an analysis of flight delays, we might only need a subset of columns.

In [24]:
delay_variables = flights.loc[:,["month","day", "carrier","origin","dest","dep_delay","arr_delay"]]
delay_variables.head()

,month,day,carrier,origin,dest,dep_delay,arr_delay
0,3,25,UA,EWR,RSW,24.0,19.0
1,4,26,DL,JFK,SFO,-4.0,-37.0
2,5,21,EV,EWR,DCA,11.0,16.0
3,7,18,EV,EWR,CLT,-8.0,-22.0
4,8,29,B6,JFK,BQN,-5.0,0.0


We can also rename variables when a clearer name would make later code easier to read.

In [25]:
delay_variables.rename(columns={"dep_delay":"departure_delay", "arr_delay":"arrival_delay"}).head()

,month,day,carrier,origin,dest,departure_delay,arrival_delay
0,3,25,UA,EWR,RSW,24.0,19.0
1,4,26,DL,JFK,SFO,-4.0,-37.0
2,5,21,EV,EWR,DCA,11.0,16.0
3,7,18,EV,EWR,CLT,-8.0,-22.0
4,8,29,B6,JFK,BQN,-5.0,0.0


<div class="alert alert-warning">
<b>Note.</b>
Selecting or renaming columns does not improve the analysis by itself. Do it when it makes the dataset easier to understand or when only a smaller set of variables is needed for the question.
</div>

## Question 4: Did flights make up time while in the air?

Sometimes the variable we need does not exist in the original dataset.

We can create a new variable from existing variables using `.assign()`.

Define:

`gain = arrival delay - departure delay`

A negative value means the flight arrived with **less delay** than it had when it departed.

In [26]:
flights_with_gain = flights.assign(gain = flights["arr_delay"] - flights["dep_delay"])
flights_with_gain.head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour,gain
0,2013,3,25,1929.0,1905,24.0,2236.0,2217,19.0,UA,1471,N37298,EWR,RSW,169.0,1068,19,5,2013-03-25 19:00:00,-5.0
1,2013,4,26,956.0,1000,-4.0,1257.0,1334,-37.0,DL,1765,N717TW,JFK,SFO,337.0,2586,10,0,2013-04-26 10:00:00,-33.0
2,2013,5,21,1320.0,1309,11.0,1430.0,1414,16.0,EV,4129,N11536,EWR,DCA,39.0,199,13,9,2013-05-21 13:00:00,5.0
3,2013,7,18,1222.0,1230,-8.0,1357.0,1419,-22.0,EV,5796,N13958,EWR,CLT,77.0,529,12,30,2013-07-18 12:00:00,-14.0
4,2013,8,29,540.0,545,-5.0,921.0,921,0.0,B6,939,N535JB,JFK,BQN,198.0,1576,5,45,2013-08-29 05:00:00,5.0


We can create several new variables at the same time.

For example, approximate average speed in miles per hour can be calculated from `distance` and `air_time`.

In [28]:
flights_transformed = flights.assign(gain = flights["arr_delay"] - flights["dep_delay"], speed=flights["distance"] / (flights["air_time"] / 60))
flights_transformed.head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,...,tailnum,origin,dest,air_time,distance,hour,minute,time_hour,gain,speed
0,2013,3,25,1929.0,1905,24.0,2236.0,2217,19.0,UA,...,N37298,EWR,RSW,169.0,1068,19,5,2013-03-25 19:00:00,-5.0,379.171598
1,2013,4,26,956.0,1000,-4.0,1257.0,1334,-37.0,DL,...,N717TW,JFK,SFO,337.0,2586,10,0,2013-04-26 10:00:00,-33.0,460.415430
2,2013,5,21,1320.0,1309,11.0,1430.0,1414,16.0,EV,...,N11536,EWR,DCA,39.0,199,13,9,2013-05-21 13:00:00,5.0,306.153846
3,2013,7,18,1222.0,1230,-8.0,1357.0,1419,-22.0,EV,...,N13958,EWR,CLT,77.0,529,12,30,2013-07-18 12:00:00,-14.0,412.207792
4,2013,8,29,540.0,545,-5.0,921.0,921,0.0,B6,...,N535JB,JFK,BQN,198.0,1576,5,45,2013-08-29 05:00:00,5.0,477.575758


### Exercise 3

Create a DataFrame called `flights_delay_change` containing a new variable:

`delay_change = arr_delay - dep_delay`

Then keep only flights where `delay_change <= -30`.

These are flights that reduced their delay by at least 30 minutes between departure and arrival.

Display the 10 flights with the largest reduction in delay.

In [21]:
flights_delay_change= flights.assign(delay_change = flights["arr_delay"] - flights["dep_delay"])
flights_delay_change = flights_delay_change.query("delay_change <= -30")
flights_delay_change.sort_values("delay_change", ascending=True).head(10)

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour,delay_change
62257,2013,6,13,1907.0,1512,235.0,2134.0,1928,126.0,EV,4377,N19554,EWR,JAX,126.0,820,15,12,2013-06-13 15:00:00,-109.0
10546,2013,2,26,1000.0,900,60.0,1513.0,1540,-27.0,HA,51,N382HA,JFK,HNL,584.0,4983,9,0,2013-02-26 09:00:00,-87.0
127816,2013,2,23,1226.0,900,206.0,1746.0,1540,126.0,HA,51,N389HA,JFK,HNL,599.0,4983,9,0,2013-02-23 09:00:00,-80.0
167806,2013,5,13,1917.0,1900,17.0,2149.0,2251,-62.0,DL,1465,N721TW,JFK,SFO,313.0,2586,19,0,2013-05-13 19:00:00,-79.0
102639,2013,2,27,924.0,900,24.0,1448.0,1540,-52.0,HA,51,N389HA,JFK,HNL,589.0,4983,9,0,2013-02-27 09:00:00,-76.0
5530,2013,7,14,1917.0,1829,48.0,2109.0,2135,-26.0,UA,673,N817UA,EWR,SNA,274.0,2434,18,29,2013-07-14 18:00:00,-74.0
55943,2013,5,2,1947.0,1949,-2.0,2209.0,2324,-75.0,UA,612,N851UA,EWR,LAX,300.0,2454,19,49,2013-05-02 19:00:00,-73.0
108369,2013,11,13,2024.0,2015,9.0,2251.0,2354,-63.0,DL,427,N188DN,JFK,LAX,311.0,2475,20,15,2013-11-13 20:00:00,-72.0
9831,2013,5,2,1949.0,1910,39.0,2208.0,2240,-32.0,AA,21,N339AA,JFK,LAX,301.0,2475,19,10,2013-05-02 19:00:00,-71.0
15531,2013,5,4,1816.0,1820,-4.0,2017.0,2131,-74.0,AS,7,N551AS,EWR,SEA,281.0,2402,18,20,2013-05-04 18:00:00,-70.0


## Question 5: What is a typical delay?

`.agg()` allows us to collapse many observations into summary statistics.

Without grouping, the summary describes the **entire dataset**.

In [33]:
flights.agg({"dep_delay": ["mean", "min", "max"], "arr_delay": ["mean", "min", "max"]})
flights_clean = flights.dropna(subset=["dep_delay", "arr_delay"])
flights_clean.agg(
    mean_dep_delay=("dep_delay", "mean"),
    median_dep_delay=("dep_delay", "median"),
    mean_arr_delay=("arr_delay", "mean"),
    median_arr_delay=("arr_delay", "median")
)

,dep_delay,arr_delay
mean_dep_delay,12.537939,NaN
median_dep_delay,-2.000000,NaN
mean_arr_delay,NaN,6.944759
median_arr_delay,NaN,-5.000000


## Question 6: Does delay differ between airports or airlines?

Usually we want summaries **within groups**.

`groupby()` changes the unit of analysis.

Instead of one row representing one flight, the resulting table can have one row representing one airport, airline, month, destination, or another group.

In [34]:
delay_by_origin = flights.groupby("origin").agg(mean_dep_delay = ("dep_delay", "mean"), mean_arr_delay = ("arr_delay", "mean"))
delay_by_origin.head()

,mean_dep_delay,mean_arr_delay
origin,,
EWR,15.083630,9.136757
JFK,12.109672,5.638486
LGA,10.328891,5.817045


This is an important conceptual change:

> Before `groupby()`: one row = one flight  
> After `groupby()` + `agg()`: one row = one origin airport

Always know what **one row represents** after a transformation.

### Exercise 4

Calculate the following for each airline (`carrier`):

* `n_flights`: number of flights;
* `avg_dep_delay`: average departure delay;
* `avg_arr_delay`: average arrival delay.

Keep only airlines with at least 1,000 flights and sort them from the **lowest to highest average arrival delay**.

In [40]:
# n_flights_per_carrier = flights.groupby("carrier").agg(n_flights = ("carrier", "count"))
# n_flights_per_carrier.head()
# avg_dep_delay = flights.groupby("carrier").agg(avg_dep_delay = ("dep_delay", "mean"))
agg_by_carrier = flights.groupby("carrier").agg(n_flights = ("carrier", "size"), mean_dep_delay = ("dep_delay", "mean"), mean_arr_delay = ("arr_delay", "mean")).query("n_flights >= 1000").sort_values("mean_arr_delay").reset_index() 
agg_by_carrier.head()


,carrier,n_flights,mean_dep_delay,mean_arr_delay
0,AA,19563,8.406411,0.383085
1,VX,3109,12.668714,1.578196
2,DL,28774,9.363554,1.848063
3,US,12384,3.866450,2.287291
4,UA,35205,11.869524,3.323555


## Combining transformations

Real analytical questions usually require more than one operation.

Method chaining lets us read the analysis as a sequence:

1. start with the data;
2. create or modify variables;
3. group observations;
4. calculate summaries;
5. filter the summaries;
6. order the result.

This mirrors the analytical workflow more closely than learning each method separately.

In [42]:
agg_by_carrier = flights.groupby("carrier").agg(n_flights = ("carrier", "size"), mean_dep_delay = ("dep_delay", "mean"), mean_arr_delay = ("arr_delay", "mean")).query("n_flights >= 1000 and mean_dep_delay > 5 and n_flights<10000").sort_values("mean_arr_delay").reset_index() 
agg_by_carrier.head()

,carrier,n_flights,mean_dep_delay,mean_arr_delay
0,VX,3109,12.668714,1.578196
1,WN,7394,17.047298,9.123862
2,FL,1962,18.042144,19.608877


### Final Exercise — Average speed by destination

Imagine you want to know how fast flights travel on average depending on destination.

Create a new variable:

`speed = distance / (air_time / 60)`

Then, for each destination (`dest`), calculate:

* `count`: number of flights;
* `avg_speed`: average speed in miles per hour.

Keep only destinations with **more than 20 flights** and sort them by `avg_speed` from fastest to slowest.

Display the 10 fastest destinations.

Try to write this as a single method chain.

In [ ]:
(
    flights.assign(speed = flights["distance"] / (flights["air_time"] / 60))
    .groupby("dest", dropna=True)
    .agg(count = ("dest", "count"), avg_speed = ("speed", "mean"))
    .query("count > 20")
    .reset_index()
    .sort_values("avg_speed", ascending=False).head(10)  
)


,count,avg_speed
dest,,
BQN,547,486.941156
SJU,3521,485.500289
HNL,428,483.518542
PSE,229,481.060702
STT,289,478.681312
LAX,9687,452.985962
SMF,164,451.506080
SAN,1647,451.474604
LGB,390,449.623567


## Check the result

A transformation is not finished just because the code ran.

Before using the result, ask:

* Does one row now represent what I think it represents?
* Are the number of groups plausible?
* Are the summary values plausible?
* Did missing values affect the calculation?
* Did filtering happen before or after aggregation as intended?

A few simple checks can prevent incorrect conclusions.

## All Done!

In this session we used pandas transformations to answer analytical questions.

We practised:

* filtering observations with `query()`;
* sorting observations with `sort_values()`;
* selecting variables with `loc[]`;
* renaming variables;
* creating new variables with `assign()`;
* grouping observations with `groupby()`;
* calculating summaries with `agg()`;
* combining several operations using method chaining;
* checking that transformed results still make analytical sense.

The key idea is:

> **Start with the question, then decide what data transformation is required.**

We will now move on to the structure of datasets and the principles of **tidy data**.